In [11]:
%matplotlib tk 

In [1]:
import torch
print("torch", torch.__version__)
print("cuda available:", torch.cuda.is_available())


torch 2.6.0+cu124
cuda available: True


In [ ]:

from sam2.build_sam import build_sam2
from sam2.sam2_image_predictor import SAM2ImagePredictor
import torch

config_path = r"C:\Projects\tivora\configs\sam2.1\sam2.1_hiera_s.yaml"  # Changed to 2.1
ckpt_path   = r"C:\Projects\tivora\checkpoints\sam2.1_hiera_small.pt"
device = "cuda" if torch.cuda.is_available() else "cpu"

sam = build_sam2(config_file=config_path, ckpt_path=ckpt_path, device=device)
predictor = SAM2ImagePredictor(sam)
print("SAM2.1 loaded on", device)


SAM2.1 loaded on cuda


In [21]:
import cv2
import numpy as np


# ---- Load person image ----
img_path = r"C:\Projects\tivora\vto_mvp\notebooks\sample01.jpeg"
image_bgr = cv2.imread(img_path)
image_rgb = cv2.cvtColor(image_bgr, cv2.COLOR_BGR2RGB)

# ---- Set image ----
predictor.set_image(image_rgb)

# ---- Give one point in T-shirt area ----
point = np.array([[image_rgb.shape[1]//2, image_rgb.shape[0]//2]])  # mid body
label = np.array([1])  # foreground

masks, scores, logits = predictor.predict(
    point_coords=point,
    point_labels=label,
    multimask_output=True
)

print("Masks returned:", len(masks))

# Pick best mask
best_id = np.argmax(scores)
tshirt_mask = masks[best_id]

# ---- Save mask ----
cv2.imwrite("tshirt_mask.png", (tshirt_mask * 255).astype("uint8"))
print("Mask saved as tshirt_mask.png")


Masks returned: 3
Mask saved as tshirt_mask.png


In [22]:
masked = image_rgb.copy()
masked[tshirt_mask == 0] = 255  # white background

cv2.imwrite("tshirt_segmented.png", cv2.cvtColor(masked, cv2.COLOR_RGB2BGR))


True

In [23]:

# Load mask & image
mask = cv2.imread("tshirt_mask.png", 0)  # grayscale
img  = cv2.imread("tshirt_segmented.png")

# Find bounding box
ys, xs = np.where(mask > 0)

top, bottom = ys.min(), ys.max()
left, right = xs.min(), xs.max()

# Crop region
crop = img[top:bottom, left:right]
mask_crop = mask[top:bottom, left:right]

cv2.imwrite("tshirt_crop.png", crop)
cv2.imwrite("tshirt_mask_crop.png", mask_crop)
print("Cropped T-shirt region saved.")

Cropped T-shirt region saved.


In [24]:
import matplotlib.pyplot as plt
import numpy as np
from PIL import Image

# Enable interactive mode
plt.ion()

img = np.array(Image.open("tshirt_crop.png"))

plt.figure(figsize=(6,8))
plt.imshow(img)
plt.title("Click 7 keypoints:\nShoulder L, Shoulder R, Chest L, Chest R, Center, Waist L, Waist R")
plt.axis('off')  # Optional: hide axes
plt.tight_layout()
plt.show(block=False)  # Show without blocking
plt.pause(0.1)  # Give it time to render

print("Click 7 points on the image...")
points = plt.ginput(7, timeout=0)  # timeout=0 means wait indefinitely
plt.close()

points = np.array(points)
np.save("user_tshirt_points.npy", points)
print("Saved user keypoints:", points)

Click 7 points on the image...
Saved user keypoints: [[ 26.46491228  13.6497076 ]
 [123.64035088  20.77251462]
 [ 38.92982456  58.16725146]
 [ 98.71052632  59.18479532]
 [ 70.98245614 107.77251462]
 [ 22.14035088 160.17602339]
 [113.71929825 150.0005848 ]]


In [ ]:
g = cv2.imread(r"C:\Projects\tivora\vto_mvp\notebooks\tshirt02.png") 
h, w = g.shape[:2]

# 7 AUTO KEYPOINTS in the correct order
# garment_points = np.array([
#     [0.23*w, 0.20*h],  # Shoulder L
#     [0.77*w, 0.20*h],  # Shoulder R
#     [0.28*w, 0.45*h],  # Chest L
#     [0.72*w, 0.45*h],  # Chest R
#     [0.50*w, 0.45*h],  # Center Chest
#     [0.32*w, 0.82*h],  # Waist L
#     [0.68*w, 0.82*h],  # Waist R
# ])
garment_points = np.array([
    [110, 120],   # Shoulder L
    [390, 125],   # Shoulder R
    [150, 200],   # Chest L
    [350, 200],   # Chest R
    [250, 260],   # Center Chest
    [160, 360],   # Waist L
    [340, 360],   # Waist R
])

np.save("garment_points.npy", garment_points)
print("Garment points:\n", garment_points)


Garment points:
 [[110 120]
 [390 125]
 [150 200]
 [350 200]
 [250 260]
 [160 360]
 [340 360]]


In [32]:
import numpy as np

user = np.load(r"C:\Projects\tivora\vto_mvp\notebooks\user_tshirt_points.npy")
garment = np.load(r"C:\Projects\tivora\vto_mvp\notebooks\garment_points.npy")

# user shoulders = points 0 & 1
user_shoulder_width = np.linalg.norm(user[1] - user[0])
garment_shoulder_width = np.linalg.norm(garment[1] - garment[0])

scale = user_shoulder_width / garment_shoulder_width
print("Scale factor:", scale)

garment_scaled = garment * scale
np.save("garment_points_scaled.npy", garment_scaled)


Scale factor: 0.34793072431488065


In [33]:
translation = user[0] - garment_scaled[0]   # align left shoulder exactly

garment_transformed = garment_scaled + translation
np.save("garment_points_transformed.npy", garment_transformed)


In [ ]:
from skimage.transform import PiecewiseAffineTransform, warp
import cv2
import numpy as np

# Load user crop for shape reference
user_img = cv2.imread(r"C:\Projects\tivora\vto_mvp\notebooks\tshirt_crop.png")

# Load garment
garment_img = cv2.imread(r"C:\Projects\tivora\vto_mvp\notebooks\tshirt02.png", cv2.IMREAD_UNCHANGED)

# Resize garment to similar height before TPS
garment_img = cv2.resize(garment_img, (user_img.shape[1], user_img.shape[0]))

# Load keypoints
user_pts = np.load(r"C:\Projects\tivora\vto_mvp\notebooks\user_tshirt_points.npy")
garment_pts = np.load(r"C:\Projects\tivora\vto_mvp\notebooks\garment_points.npy")

# TPS Warp
tps = PiecewiseAffineTransform()
tps.estimate(garment_pts, user_pts)

warped = warp(
    garment_img,
    tps,
    output_shape=user_img.shape[:2],
    mode="constant",
    cval=0
)

warped = (warped * 255).astype(np.uint8)
cv2.imwrite("warped_garment.png", warped)
print("Warped garment saved.")


Warped garment saved.
